# Homework: Logistic Regression
## SCIS 432: Artificial Intelligence

In this assignment, you will implement and explore **logistic regression** from the ground up:

- Understanding the **Bernoulli and binomial distributions**
- Working with **odds, log-odds, and the sigmoid function**
- Deriving and computing **binary cross-entropy loss**
- Deriving the **gradient of cross-entropy** through the sigmoid
- Understanding the connection between **linear and logistic regression**
- Fitting a logistic regression model with **sklearn**
- Evaluating classification with **accuracy, precision, recall, F1, and the confusion matrix**
- Implementing logistic regression from scratch using **gradient descent**

---
### Submission
Save as a PDF and submit with all code cells run and all written answers filled in.

### Rules
- You may use `numpy`, `matplotlib`, and `sklearn`.
- In Parts 1–5 (conceptual), no coding is required — answer in the Markdown cells provided.
- In Parts 6–7, you may use `sklearn.linear_model.LogisticRegression` and `sklearn.model_selection.train_test_split`.
- In Part 8, you will implement logistic regression **from scratch** using only `numpy` — do not call any sklearn fitting functions.

## Learning Objectives

By the end of this assignment, you should be able to:

- Define the Bernoulli and binomial distributions and compute probabilities by hand
- Convert between probability, odds, and log-odds
- Derive and implement the sigmoid function and compute its derivative
- Derive the binary cross-entropy loss from first principles (maximum likelihood)
- Compute gradients of cross-entropy with respect to model parameters
- Explain how logistic regression extends linear regression to classification
- Fit and evaluate a logistic regression model using sklearn
- Implement logistic regression with gradient descent from scratch

## Setup — Run This First

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Load the breast cancer dataset
cancer = load_breast_cancer()
X_all = cancer.data      # shape (569, 30)
y_all = cancer.target    # 0 = malignant, 1 = benign

print("Dataset: sklearn breast cancer")
print(f"  X shape : {X_all.shape}")
print(f"  y shape : {y_all.shape}")
print(f"  Classes : {cancer.target_names}  (0=malignant, 1=benign)")
print(f"  Class counts: malignant={np.sum(y_all==0)}, benign={np.sum(y_all==1)}")
print(f"  Features (first 5): {cancer.feature_names[:5]}")

---
## Part 1 — The Bernoulli and Binomial Distributions

Logistic regression models **binary outcomes** — events that are either 0 or 1. Before building the model, we need to understand the probability distributions that govern such outcomes.

### The Bernoulli Distribution

A **Bernoulli random variable** $Y \sim \text{Bernoulli}(p)$ takes the value 1 with probability $p$ and 0 with probability $1-p$. Its probability mass function is:

$$P(Y = y) = p^y (1-p)^{1-y}, \quad y \in \{0, 1\}$$

Its mean and variance are:

$$\mathbb{E}[Y] = p \qquad \text{Var}(Y) = p(1-p)$$

### The Binomial Distribution

If we run $n$ **independent** Bernoulli trials each with success probability $p$, the total number of successes $S \sim \text{Binomial}(n, p)$ has PMF:

$$P(S = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n$$

$$\mathbb{E}[S] = np \qquad \text{Var}(S) = np(1-p)$$

### 1.1 — Written Questions

**Answer each question in the Markdown cell below.**

1. A diagnostic test for a disease has a **sensitivity** (true positive rate) of 0.85. This means that for a patient who truly has the disease, the test returns positive with probability 0.85. Model a single test result for a sick patient as a Bernoulli random variable. What is $p$? What is the variance of a single test outcome?

2. The same test is administered to **20 sick patients independently**. Let $S$ be the number of positive results.
   - What is the distribution of $S$? State it fully (name and parameters).
   - Compute $P(S = 20)$ (all 20 test positive). Show your work.
   - Compute $P(S \geq 18)$ (at least 18 test positive). Show your work. *(Hint: $P(S \geq 18) = P(S=18) + P(S=19) + P(S=20)$.)*
   - What is $\mathbb{E}[S]$ and $\text{Var}(S)$?

3. The Bernoulli PMF is written $P(Y=y) = p^y(1-p)^{1-y}$. Verify that this formula gives the correct probability for both $y=0$ and $y=1$ by substituting each value. Why is this compact form useful?

4. For the Bernoulli distribution, at what value of $p$ is the variance maximized? At what values is it minimized? Give a brief intuitive explanation for why variance behaves this way.

**Your answers here:**

1.

2.

3.

4.

---
## Part 2 — Odds, Log-Odds, and the Sigmoid Function

### From Probability to Odds

Given a probability $p \in (0, 1)$, the **odds** of the event are:

$$\text{odds} = \frac{p}{1 - p}$$

Odds range from 0 to $\infty$. Odds $> 1$ mean the event is more likely to occur than not.

### Log-Odds (Logit)

The **log-odds** (also called the **logit**) is:

$$\text{logit}(p) = \log\frac{p}{1-p}$$

The log-odds map $p \in (0,1)$ to $\mathbb{R}$ — the entire real line. This is convenient because linear models naturally produce real-valued outputs.

### The Sigmoid Function

The **sigmoid** (or logistic) function is the inverse of the logit. It maps any real number back to a probability in $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

If we let $z = \mathbf{w}^\top \mathbf{x} + b$ be a linear combination of inputs (as in linear regression), then $\hat{p} = \sigma(z)$ gives us the **predicted probability** of class 1.

### 2.1 — Written Questions

1. Convert each probability to odds and then to log-odds. Fill in the table below. Show your work for at least one row.

   | Probability $p$ | Odds $\frac{p}{1-p}$ | Log-odds $\log\frac{p}{1-p}$ |
   |:-:|:-:|:-:|
   | 0.10 | | |
   | 0.25 | | |
   | 0.50 | | |
   | 0.75 | | |
   | 0.90 | | |

2. Looking at the completed table: what happens to the log-odds when $p = 0.5$? What does the symmetry in the log-odds values tell you about the relationship between $p$ and $1-p$?

3. A model outputs a log-odds of $z = 2.2$. Compute the predicted probability $\hat{p} = \sigma(2.2)$ using the sigmoid formula. Show your work (you may use a calculator).

4. **Derivative of the sigmoid.** Show algebraically that:
   $$\frac{d\sigma}{dz} = \sigma(z)\bigl(1 - \sigma(z)\bigr)$$
   *Hint: Write $\sigma(z) = (1 + e^{-z})^{-1}$, differentiate using the chain rule, then factor the result in terms of $\sigma(z)$.*

5. Using the result from (4), at what value of $z$ is $\frac{d\sigma}{dz}$ maximized? What is that maximum value? What does this tell you about where the sigmoid is changing most rapidly?

**Your answers here:**

1.

2.

3.

4.

5.

### 2.2 — Implement and Plot the Sigmoid

In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid function element-wise.
    Works for scalars and numpy arrays.
    sigma(z) = 1 / (1 + exp(-z))
    """
    # TODO
    raise NotImplementedError


def sigmoid_derivative(z):
    """
    Compute d(sigma)/dz = sigma(z) * (1 - sigma(z)) element-wise.
    Hint: reuse your sigmoid function.
    """
    # TODO
    raise NotImplementedError


# --- Plot sigmoid and its derivative ---
z_vals = np.linspace(-8, 8, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z_vals, sigmoid(z_vals), color='steelblue', linewidth=2)
axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='y=0.5')
axes[0].axvline(0,   color='gray', linestyle='--', linewidth=1, label='z=0')
axes[0].set_xlabel('z')
axes[0].set_ylabel('sigma(z)')
axes[0].set_title('Sigmoid Function')
axes[0].legend()

axes[1].plot(z_vals, sigmoid_derivative(z_vals), color='tomato', linewidth=2)
axes[1].set_xlabel('z')
axes[1].set_ylabel("sigma'(z)")
axes[1].set_title("Sigmoid Derivative")

plt.tight_layout()
plt.show()

# Sanity check
print(f"sigma(0)   = {sigmoid(0):.4f}  (expected 0.5000)")
print(f"sigma(2.2) = {sigmoid(2.2):.4f}")
print(f"sigma'(0)  = {sigmoid_derivative(0):.4f}  (expected 0.2500)")

---
## Part 3 — From Linear to Logistic Regression

**Linear regression** models the expected value of a continuous output directly as a linear function of the inputs:

$$\hat{y} = \mathbf{w}^\top \mathbf{x} + b$$

**Logistic regression** handles binary classification by modeling the **probability** of class 1. The key idea is to pass the linear output through the sigmoid:

$$z = \mathbf{w}^\top \mathbf{x} + b \qquad \hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

This model assumes that the **log-odds** of the outcome are a linear function of the features:

$$\log\frac{\hat{p}}{1-\hat{p}} = \mathbf{w}^\top \mathbf{x} + b$$

We classify as class 1 when $\hat{p} \geq 0.5$, which is equivalent to $z \geq 0$.

### 3.1 — Written Questions

1. **Why can't we use linear regression directly for binary classification?** Give two distinct reasons. *(Think about the range of linear regression outputs and what happens to the squared-error loss for a binary target.)*

2. **Interpreting coefficients.** In logistic regression, a coefficient $w_j$ represents the change in **log-odds** per unit increase in feature $x_j$, holding all other features constant. Suppose a model predicts the probability of a loan default, and the coefficient on the feature "number of missed payments" is $w = 0.65$.
   - By how much do the log-odds of default change for each additional missed payment?
   - The **odds ratio** is $e^{w}$. Compute $e^{0.65}$ and interpret it in plain English.
   - If a borrower currently has a predicted probability of default of 0.30, what are their current odds? After one more missed payment, what are their new odds? What is the new probability?

3. **Decision boundary.** For a logistic regression model with two features ($x_1, x_2$) and parameters $w_1 = 1.5$, $w_2 = -2.0$, $b = 0.5$:
   - Write down the equation of the **decision boundary** (the set of points where $\hat{p} = 0.5$, i.e., $z = 0$). Simplify it to the form $x_2 = ax_1 + c$.
   - Is the decision boundary for logistic regression always linear in the original feature space? What would you have to do to get a nonlinear boundary?

4. In logistic regression, what is the role of the **bias term** $b$? How does it relate to the decision boundary geometrically?

**Your answers here:**

1.

2.

3.

4.

---
## Part 4 — Binary Cross-Entropy Loss

For binary classification, we use **binary cross-entropy** (BCE) as our loss function. It arises naturally from **maximum likelihood estimation** under the Bernoulli model.

### Derivation via Maximum Likelihood

Given $N$ independent training examples $(\mathbf{x}_i, y_i)$ with $y_i \in \{0,1\}$ and predicted probabilities $\hat{p}_i = \sigma(\mathbf{w}^\top \mathbf{x}_i + b)$, the **likelihood** is:

$$L(\mathbf{w}, b) = \prod_{i=1}^{N} \hat{p}_i^{y_i}(1-\hat{p}_i)^{1-y_i}$$

Taking the negative log gives the **negative log-likelihood** (NLL), which we minimize:

$$J(\mathbf{w}, b) = -\frac{1}{N}\sum_{i=1}^{N}\Bigl[y_i \log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Bigr]$$

This is the **binary cross-entropy loss**.

### 4.1 — Written Questions

1. **Unpacking the formula.** The BCE loss has two terms inside the sum: $y_i \log \hat{p}_i$ and $(1-y_i)\log(1-\hat{p}_i)$.
   - When $y_i = 1$, which term survives and which drops out? What does the surviving term penalize?
   - When $y_i = 0$, which term survives? What does it penalize?
   - Why is this more appropriate for classification than mean squared error?

2. **Compute BCE by hand.** Suppose we have 4 training examples with true labels and predicted probabilities:

   | $i$ | $y_i$ | $\hat{p}_i$ |
   |:---:|:-----:|:-----------:|
   | 1   | 1     | 0.90        |
   | 2   | 1     | 0.60        |
   | 3   | 0     | 0.20        |
   | 4   | 0     | 0.85        |

   Compute the BCE loss for each example individually, then compute the mean. Show all work. *(You may use $\log(0.9) \approx -0.105$, $\log(0.6) \approx -0.511$, $\log(0.8) \approx -0.223$, $\log(0.4) \approx -0.916$, $\log(0.15) \approx -1.897$.)*

3. **Perfect and terrible predictions.** Suppose a model is perfectly confident and correct for every example ($\hat{p}_i = 1$ when $y_i=1$ and $\hat{p}_i = 0$ when $y_i=0$).
   - What is the BCE loss in this case?
   - What happens to the loss when the model is perfectly confident but **wrong** (e.g., $y_i=1$ but $\hat{p}_i \to 0$)? Why does this make sense as a penalty?

4. **Relationship to entropy.** The term "cross-entropy" comes from information theory. The entropy of the true distribution $p^* \in \{0, 1\}$ is zero (it is deterministic). Briefly explain in your own words why minimizing cross-entropy loss is equivalent to making the model's predicted distribution as close as possible to the true distribution.

**Your answers here:**

1.

2.

3.

4.

---
## Part 5 — Gradients of the Cross-Entropy Loss

To train logistic regression with gradient descent, we need the gradient of the BCE loss with respect to the weights $\mathbf{w}$ and bias $b$.

Consider a single example $(\mathbf{x}, y)$ with $z = \mathbf{w}^\top \mathbf{x} + b$ and $\hat{p} = \sigma(z)$. The per-example loss is:

$$\ell = -\bigl[y\log\hat{p} + (1-y)\log(1-\hat{p})\bigr]$$

We will derive $\frac{\partial \ell}{\partial w_j}$ step by step using the chain rule:

$$\frac{\partial \ell}{\partial w_j} = \frac{\partial \ell}{\partial \hat{p}} \cdot \frac{\partial \hat{p}}{\partial z} \cdot \frac{\partial z}{\partial w_j}$$

### 5.1 — Derive the Gradient (Written)

Work through each step of the chain rule derivation.

1. **Step 1.** Compute $\frac{\partial \ell}{\partial \hat{p}}$, the derivative of the per-example BCE loss with respect to the predicted probability $\hat{p}$. Show your algebra.

2. **Step 2.** We showed in Part 2 that $\frac{\partial \hat{p}}{\partial z} = \hat{p}(1-\hat{p})$. Write this down.

3. **Step 3.** Compute $\frac{\partial z}{\partial w_j}$ where $z = \mathbf{w}^\top \mathbf{x} + b$.

4. **Chain rule.** Multiply the three pieces together. You should get a surprisingly clean result. Simplify fully and state what you notice about the final gradient. *(Hint: several terms should cancel.)*

5. **Full dataset.** Extending to $N$ examples, the average gradient is:
   $$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{N}\sum_{i=1}^{N}(\hat{p}_i - y_i)\mathbf{x}_i \qquad \frac{\partial \mathcal{J}}{\partial b} = \frac{1}{N}\sum_{i=1}^{N}(\hat{p}_i - y_i)$$
   Compare this to the gradient of **MSE loss** in linear regression: $\frac{\partial L_{\text{MSE}}}{\partial \mathbf{w}} = \frac{-2}{N}\sum_i(y_i - \hat{y}_i)\mathbf{x}_i$. What do you notice about the structural similarity? What is the key difference?

**Your answers here:**

1.

2.

3.

4.

5.

---
## Part 6 — Data Preparation

Before fitting any model we need to:
1. Split the data into training and test sets.
2. **Standardize** the features — logistic regression (and especially gradient descent) is sensitive to feature scale.

Feature standardization transforms each feature $x_j$ to have zero mean and unit variance:
$$x_j' = \frac{x_j - \mu_j}{\sigma_j}$$

**Important:** Fit the scaler **only on the training set**, then apply it to both train and test. Fitting on the test set would be **data leakage**.

In [ ]:
# TODO: Split X_all and y_all into X_train, X_test, y_train, y_test
# Use test_size=0.20 and random_state=42 and stratify=y_all
# (stratify ensures both splits preserve the class proportions)

# X_train, X_test, y_train, y_test = ...
raise NotImplementedError

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
print(f"Train class balance: {np.mean(y_train):.3f} (fraction benign)")
print(f"Test  class balance: {np.mean(y_test):.3f} (fraction benign)")

In [ ]:
# TODO: Standardize the features using StandardScaler.
# 1. Create a StandardScaler object.
# 2. Call .fit_transform(X_train) to get X_train_scaled.
# 3. Call .transform(X_test)  to get X_test_scaled  (do NOT re-fit on test!).

# scaler = ...
# X_train_scaled = ...
# X_test_scaled  = ...
raise NotImplementedError

print(f"X_train_scaled mean (should be ~0): {X_train_scaled.mean(axis=0)[:3].round(4)}")
print(f"X_train_scaled std  (should be ~1): {X_train_scaled.std(axis=0)[:3].round(4)}")

### 6.1 — Written Questions

1. What is **data leakage**, and why does fitting the scaler on the test set constitute leakage?
2. What is the `stratify` parameter in `train_test_split` doing, and when is it especially important to use?
3. Logistic regression with gradient descent is sensitive to feature scale. Explain intuitively why — consider what happens to the gradient when one feature has values in the range $[0, 10000]$ and another in $[0, 1]$.

**Your answers here:**

1.

2.

3.

---
## Part 7 — Fitting with sklearn and Evaluation

Now we will fit a logistic regression model using sklearn and thoroughly evaluate its performance.

For binary classification, we go beyond accuracy. Key metrics:

| Metric | Formula | Meaning |
|--------|---------|----------|
| **Accuracy** | $\frac{TP + TN}{N}$ | Overall fraction correct |
| **Precision** | $\frac{TP}{TP + FP}$ | Of predicted positives, how many are correct? |
| **Recall** | $\frac{TP}{TP + FN}$ | Of actual positives, how many were caught? |
| **F1 Score** | $\frac{2 \cdot \text{Prec} \cdot \text{Rec}}{\text{Prec} + \text{Rec}}$ | Harmonic mean of precision and recall |

where TP = true positives, TN = true negatives, FP = false positives, FN = false negatives.

In our dataset: **positive = benign (1)**, **negative = malignant (0)**.

In [ ]:
# TODO: Fit a LogisticRegression model on X_train_scaled, y_train.
# Use max_iter=1000 and random_state=42. Leave all other parameters at defaults.

# lr_model = ...
# lr_model.fit(...)
raise NotImplementedError

# TODO: Generate predictions on train and test sets
# y_train_pred = ...
# y_test_pred  = ...

# TODO: Also get predicted probabilities (for the positive class)
# y_test_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

raise NotImplementedError

In [ ]:
# TODO: Compute and print the following for BOTH train and test sets:
#   - Accuracy
#   - Precision
#   - Recall
#   - F1 Score
# Then print the confusion matrix for the test set.
# Finally, print sklearn's classification_report for the test set.

# Hint: use accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
# imported at the top of the notebook.

raise NotImplementedError

In [ ]:
# TODO: Plot the confusion matrix as a heatmap using matplotlib.
# Label the axes clearly with 'Predicted' and 'Actual',
# and use the class names ['Malignant', 'Benign'] for tick labels.
# Annotate each cell with its count.
#
# Hint:
#   cm = confusion_matrix(y_test, y_test_pred)
#   plt.imshow(cm, cmap='Blues')
#   for i in range(2):
#       for j in range(2):
#           plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)

raise NotImplementedError

In [ ]:
# TODO: Retrieve the model's learned coefficients and bias.
# Plot a horizontal bar chart of the coefficients for all 30 features,
# sorted from most negative to most positive.
# Use cancer.feature_names as the y-axis labels.
#
# Hint:
#   coefs = lr_model.coef_[0]          # shape (30,)
#   bias  = lr_model.intercept_[0]

raise NotImplementedError

### 7.1 — Written Questions

1. Report the test **accuracy, precision, recall, and F1**. In the context of cancer diagnosis, which metric do you think is most important to maximize, and why? Consider the consequences of false positives vs. false negatives.

2. Look at your confusion matrix. How many malignant tumors were misclassified as benign (false negatives)? How many benign were misclassified as malignant (false positives)? Which type of error is more serious here?

3. The default decision threshold is $\hat{p} \geq 0.5$. If you **lowered** the threshold to $\hat{p} \geq 0.3$, what would happen to:
   - The number of true positives and false negatives?
   - Recall? Precision? Overall accuracy?
   - Would this be a good idea for cancer screening? Explain.

4. Look at the coefficient bar chart. Identify the two features with the largest **positive** coefficients and the two with the largest **negative** coefficients. What does a large positive coefficient mean for the predicted probability of the positive class (benign)?

5. The train accuracy is likely close to (or slightly above) the test accuracy. What would it mean if the train accuracy were 0.99 but the test accuracy were 0.85? What is this phenomenon called?

**Your answers here:**

1.

2.

3.

4.

5.

---
## Part 8 — Logistic Regression from Scratch with Gradient Descent

Now you will implement logistic regression yourself using only `numpy`. We derived in Part 5 that the gradients of the BCE loss are:

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{N}\sum_{i=1}^{N}(\hat{p}_i - y_i)\mathbf{x}_i \qquad \frac{\partial J}{\partial b} = \frac{1}{N}\sum_{i=1}^{N}(\hat{p}_i - y_i)$$

In matrix notation (vectorized over all $N$ examples at once):

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{N} X^\top (\hat{\mathbf{p}} - \mathbf{y}) \qquad \frac{\partial J}{\partial b} = \frac{1}{N} \mathbf{1}^\top (\hat{\mathbf{p}} - \mathbf{y})$$

where $X$ is the $(N \times D)$ feature matrix, $\hat{\mathbf{p}}$ is the $(N,)$ vector of predicted probabilities, and $\mathbf{y}$ is the $(N,)$ vector of true labels.

In [ ]:
def binary_cross_entropy(y_true, y_pred_proba, eps=1e-12):
    """
    Compute mean binary cross-entropy loss.

    Args:
        y_true       : 1-D array of true labels (0 or 1), shape (N,)
        y_pred_proba : 1-D array of predicted probabilities, shape (N,)
        eps          : small constant to avoid log(0)

    Returns:
        loss (float): mean BCE loss

    Hint: clip predictions to [eps, 1-eps] before taking the log:
        p = np.clip(y_pred_proba, eps, 1 - eps)
    """
    # TODO
    raise NotImplementedError


# Quick sanity check
y_check  = np.array([1, 0, 1, 0])
p_perfect = np.array([0.999, 0.001, 0.999, 0.001])
p_bad     = np.array([0.001, 0.999, 0.001, 0.999])
p_random  = np.array([0.5,   0.5,   0.5,   0.5  ])

print(f"BCE (perfect predictions) : {binary_cross_entropy(y_check, p_perfect):.6f}  (should be ~0)")
print(f"BCE (all wrong)           : {binary_cross_entropy(y_check, p_bad):.4f}     (should be large)")
print(f"BCE (random 0.5)          : {binary_cross_entropy(y_check, p_random):.4f}  (should be ~0.6931)")

In [ ]:
def logistic_regression_gd(X, y, learning_rate=0.1, n_iterations=1000, random_state=42):
    """
    Fit logistic regression using gradient descent.

    Args:
        X              : feature matrix, shape (N, D)
        y              : binary labels, shape (N,)
        learning_rate  : step size alpha
        n_iterations   : number of gradient steps
        random_state   : seed for reproducible weight initialization

    Returns:
        w          : fitted weight vector, shape (D,)
        b          : fitted bias (float)
        loss_curve : list of BCE loss values, one per iteration

    Steps:
        1. Initialize w to a vector of small random values, b = 0.
               rng = np.random.default_rng(random_state)
               w = rng.normal(0, 0.01, size=X.shape[1])
               b = 0.0
        2. For each iteration:
               a. Compute z = X @ w + b                          (shape N,)
               b. Compute p_hat = sigmoid(z)                    (shape N,)
               c. Compute error = p_hat - y                     (shape N,)
               d. Compute gradients:
                      dw = (1/N) * X.T @ error                  (shape D,)
                      db = (1/N) * np.sum(error)                (scalar)
               e. Update: w -= learning_rate * dw
                          b -= learning_rate * db
               f. Record binary_cross_entropy(y, p_hat)
        3. Return w, b, loss_curve
    """
    N, D = X.shape
    rng = np.random.default_rng(random_state)
    w = rng.normal(0, 0.01, size=D)
    b = 0.0
    loss_curve = []

    for _ in range(n_iterations):
        # TODO: implement steps 2a–2f
        raise NotImplementedError

    return w, b, loss_curve


# Fit on the scaled training data
w_gd, b_gd, loss_curve = logistic_regression_gd(
    X_train_scaled, y_train, learning_rate=0.1, n_iterations=500
)

print(f"Final training BCE loss : {loss_curve[-1]:.4f}")
print(f"Bias term (b)           : {b_gd:.4f}")

In [ ]:
# TODO: Plot the BCE loss curve over iterations.
# Label axes and add a title.

raise NotImplementedError

In [ ]:
def predict_proba_lr(X, w, b):
    """
    Compute predicted probabilities for logistic regression.
    Returns sigmoid(X @ w + b), shape (N,).
    """
    # TODO
    raise NotImplementedError


def predict_lr(X, w, b, threshold=0.5):
    """
    Predict class labels: 1 if predicted probability >= threshold, else 0.
    """
    # TODO
    raise NotImplementedError


# Evaluate on train and test
train_preds_gd = predict_lr(X_train_scaled, w_gd, b_gd)
test_preds_gd  = predict_lr(X_test_scaled,  w_gd, b_gd)

print("=== From-Scratch Logistic Regression (GD) ===")
print(f"Train Accuracy : {accuracy_score(y_train, train_preds_gd):.4f}")
print(f"Test  Accuracy : {accuracy_score(y_test,  test_preds_gd):.4f}")
print(f"Test  F1 Score : {f1_score(y_test, test_preds_gd):.4f}")
print()
print("=== sklearn Logistic Regression (for comparison) ===")
print(f"Train Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Test  Accuracy : {accuracy_score(y_test,  y_test_pred):.4f}")
print(f"Test  F1 Score : {f1_score(y_test, y_test_pred):.4f}")

### 8.1 — Learning Rate Experiment

In [ ]:
# TODO: Train your from-scratch logistic regression with three different learning rates:
#   learning_rates = [0.001, 0.1, 1.0]
# Use n_iterations=500 for each.
#
# On a single plot, show the loss curve for each learning rate.
# Label each curve with its learning rate in the legend.
# Add axis labels and a title.

raise NotImplementedError

### 8.2 — Written Questions

1. How does your from-scratch implementation compare to sklearn's logistic regression in terms of test accuracy and F1? If they differ, suggest a reason why.

2. Describe what you observe in the learning rate experiment. What happens with a very small learning rate? What happens with a very large one? Is there a regime where the loss oscillates or diverges?

3. Your gradient descent implementation processes **all $N$ training examples** in each iteration (batch gradient descent). Describe **stochastic gradient descent (SGD)** and **mini-batch gradient descent**. What are the trade-offs between batch, mini-batch, and stochastic approaches in terms of convergence speed and stability?

4. Logistic regression is a **convex** optimization problem — the BCE loss has no local minima, only a global minimum. What practical implication does this have for gradient descent (compared to, say, training a neural network)?

**Your answers here:**

1.

2.

3.

4.

---
## Reflection

Answer in a few sentences each:

1. We derived that the gradient of BCE loss through the sigmoid takes the simple form $(\hat{p}_i - y_i)\mathbf{x}_i$. Reflect on why this is a satisfying result — what does it say about the role of the sigmoid and the BCE loss being "matched" to each other?

2. Logistic regression assumes a **linear decision boundary**. In what kinds of real-world problems might this be too restrictive? What are two approaches you could use to get a nonlinear decision boundary while still using logistic regression?

3. A colleague trains a logistic regression model on a dataset where 95% of examples are class 0 and 5% are class 1. The model predicts class 0 for every example and achieves 95% accuracy. What is wrong with using accuracy as the sole metric here, and what would you recommend instead?

**Your answers here:**

1.

2.

3.

---
## Grading Rubric (100 points)

| Part | Description | Points |
|------|-------------|--------|
| 1 | Bernoulli & binomial — written questions | 10 |
| 2 | Odds, log-odds, sigmoid — written questions + `sigmoid`, `sigmoid_derivative` + plots | 12 |
| 3 | Linear vs. logistic regression — written questions | 10 |
| 4 | Binary cross-entropy — written questions | 10 |
| 5 | Gradient derivation — written questions | 10 |
| 6 | Data prep (split + scale) + written questions | 8 |
| 7 | sklearn fit, all metrics, confusion matrix plot, coefficient plot, written questions | 15 |
| 8 | `binary_cross_entropy`, `logistic_regression_gd`, `predict_lr`, loss curve, LR experiment, written questions | 20 |
| Reflection | 3 reflection questions | 5 |

**Extra Credit (+10):** Implement **early stopping** in your gradient descent: after each iteration, evaluate the BCE loss on a held-out **validation set** (use 10% of the training set). Stop training when the validation loss has not decreased for 20 consecutive iterations. Compare the loss curves (train vs. validation) and report whether early stopping prevents overfitting on this dataset.